# 00. Data Understanding and Validation

This notebook documents the initial review of the original UCI **Bank Marketing** dataset. The goal is to understand the records and contact-history groups before implementing the three modeling approaches in the README.

**Scope:** local data loading, integrity and schema checks, descriptive summaries, missing-value review, history consistency, duplicate checks, and the prediction-time decisions. No data is downloaded, no raw file is changed, no model is fitted, and no train/test split is created.

**How to run:** select the portfolio's existing Python environment (`../.venv/Scripts/python.exe` from the project folder), then **Restart Kernel and Run All**. Run from this project's root or its `notebooks/` folder. The code reads the local CSV and source manifest and displays results without exporting transformed data.

**Evaluation boundary:** these are full-file descriptive summaries, including outcome distributions. They are not held-out performance estimates. A future split must be fixed before feature/model selection; disclose this exploratory exposure rather than calling the source file entirely unseen test data.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import importlib.metadata

import numpy as np
import pandas as pd
from IPython.display import display

# Display formatting changes presentation only, not the values used in calculations.
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

environment = pd.Series({
    "Python": platform.python_version(),
    "pandas": pd.__version__,
    "NumPy": np.__version__,
    "IPython": importlib.metadata.version("ipython"),
}, name="version").to_frame()
display(environment)

,version
Python,3.14.3
pandas,3.0.5
NumPy,2.5.1
IPython,9.15.0


## 1. Source, variant, and file integrity

Use **bank-full.csv**, the original 45,211-record variant with 16 inputs and the outcome `y`. Do not substitute the 4,521-row sample or the different `bank-additional` variant.

- [Official UCI dataset page](https://archive.ics.uci.edu/dataset/222/bank+marketing)
- [Dataset DOI](https://doi.org/10.24432/C5K306), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)
- Local original dictionary: `data/raw/bank-names.txt`
- Local download details and SHA-256 hashes: `reports/source_manifest.json`

The original dictionary cites Moro, Laureano, and Cortez (2011), *Using Data Mining for Bank Direct Marketing: An Application of the CRISP-DM Methodology*. It states that the full file is ordered by date, May 2008 to November 2010. We preserve that order; the CSV itself lacks a full date and customer identifier.

The checks below compare the local raw files with their recorded download hashes. They verify local integrity, not the accuracy of every original observation.

In [2]:
PROJECT_NAME = "04_bank_marketing_history_segmentation"
cwd = Path.cwd().resolve()
# Support both notebook and project-root working directories without a machine-specific path.
candidates = [cwd, cwd.parent, cwd / PROJECT_NAME]
PROJECT_ROOT = next((
    path for path in candidates
    if path.name == PROJECT_NAME and (path / "data/raw/bank-full.csv").is_file()
), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the Project 4 root or its notebooks folder; bank-full.csv must be in data/raw.")

DATA_PATH = PROJECT_ROOT / "data/raw/bank-full.csv"
MANIFEST_PATH = PROJECT_ROOT / "reports/source_manifest.json"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

raw_hashes_before = {}
integrity_records = []
# Stop if local source bytes differ from the recorded download; do not silently analyze a changed file.
for file in manifest["files"]:
    path = PROJECT_ROOT / file["path"]
    actual_hash = sha256(path)
    assert actual_hash == file["sha256"], f"Hash mismatch: {file['path']}"
    assert path.stat().st_size == file["bytes"], f"Size mismatch: {file['path']}"
    raw_hashes_before[file["path"]] = actual_hash
    integrity_records.append({"file": file["path"], "bytes": path.stat().st_size, "hash_matches": True})

# This fixed fingerprint also pins the intended original variant, independently of the manifest.
assert sha256(DATA_PATH) == "d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa264b3aaf0f849ed32d"
print("Project:", PROJECT_ROOT.name)
display(pd.DataFrame(integrity_records))

Project: 04_bank_marketing_history_segmentation


,file,bytes,hash_matches
0,data/raw/bank_marketing_uci_original.zip,1023843,True
1,data/raw/bank.zip,579043,True
2,data/raw/bank-full.csv,4610348,True
3,data/raw/bank-names.txt,3864,True


## 2. Load the data and inspect its structure

The file uses a **semicolon**, not a comma. Each row is a recorded client/campaign observation with the final subscription outcome. Without a customer ID, a row cannot be certified as a distinct person.

Two views are read: a normal typed DataFrame and a text-only view for checking literal missing-value tokens. Neither is written back to disk. `df` remains the unmodified loaded dataset.

In [3]:
expected_columns = [
    "age", "job", "marital", "education", "default", "balance", "housing", "loan",
    "contact", "day", "month", "duration", "campaign", "pdays", "previous", "poutcome", "y",
]
# The original export is semicolon-delimited; a comma would produce the wrong schema.
df = pd.read_csv(DATA_PATH, sep=";")
# Preserve literal tokens in a separate view so blank values and 'unknown' can be audited.
text_view = pd.read_csv(DATA_PATH, sep=";", dtype="string", keep_default_na=False)

assert df.shape == (45211, 17), f"Unexpected shape: {df.shape}"
assert df.columns.tolist() == expected_columns, "Unexpected schema or variant."
assert text_view.shape == df.shape
# Retain an independent snapshot to detect later changes to values, dtypes, or row order.
loaded_snapshot = df.copy(deep=True)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]} = {df.shape[1] - 1} inputs + y")
print("Preview: selected columns only; these first rows are not a representative sample.")
display(df.loc[:, ["age", "job", "education", "balance", "previous", "poutcome", "y"]].head())

Rows: 45,211
Columns: 17 = 16 inputs + y
Preview: selected columns only; these first rows are not a representative sample.


,age,job,education,balance,previous,poutcome,y
0,58,management,tertiary,2143,0,unknown,no
1,44,technician,secondary,29,0,unknown,no
2,33,entrepreneur,secondary,2,0,unknown,no
3,47,blue-collar,unknown,1506,0,unknown,no
4,33,unknown,unknown,1,0,unknown,no


In [4]:
# Parsed nulls and source tokens answer different questions; do not collapse them into one count.
column_review = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "distinct_values": df.nunique(dropna=False),
    "parsed_nulls": df.isna().sum(),
    "blank_tokens": text_view.apply(lambda col: col.str.strip().eq("").sum()),
    "question_marks": text_view.eq("?").sum(),
    "unknown_tokens": text_view.eq("unknown").sum(),
})
display(column_review)
print("Total parsed nulls:", int(df.isna().sum().sum()))
print("Columns containing the literal 'unknown':")
display(column_review.loc[column_review["unknown_tokens"].gt(0), ["unknown_tokens"]])

,dtype,distinct_values,parsed_nulls,blank_tokens,question_marks,unknown_tokens
age,int64,77,0,0,0,0
job,str,12,0,0,0,288
marital,str,3,0,0,0,0
education,str,4,0,0,0,1857
default,str,2,0,0,0,0
balance,int64,7168,0,0,0,0
housing,str,2,0,0,0,0
loan,str,2,0,0,0,0
contact,str,3,0,0,0,13020
day,int64,31,0,0,0,0


Total parsed nulls: 0
Columns containing the literal 'unknown':


,unknown_tokens
job,288
education,1857
contact,13020
poutcome,36959


### Interpretation: no parsed nulls does not mean complete information

The original file has no parsed nulls, blank tokens, or `?` tokens in these checks. The literal category `unknown` appears in `job`, `education`, `contact`, and `poutcome` and must be considered separately.

- Do not globally replace every `unknown` with failure, zero, or a generic numeric value.
- `pdays == -1` denotes no previous campaign contact in the source dictionary, not a negative elapsed time to impute. It is checked for consistency below, but `pdays` is excluded from primary models.
- Missing-value handling, if later needed, must be defined within the training workflow. This notebook does not impute or clean observations.

## 3. Outcome distribution

The target `y` records whether the client subscribed to a term deposit: `yes` or `no`. The in-memory binary indicator below is only for descriptive summaries.

The positive rate describes this historically contacted population. It is not the probability that a new outreach action causes a subscription.

In [5]:
# Validate labels first so an unexpected value cannot silently become a negative outcome.
assert set(df["y"].unique()) == {"no", "yes"}, "Unexpected target values."
# A binary indicator makes its mean the observed subscription rate; it is not a model score.
subscribed = df["y"].eq("yes").astype("int8")
target_summary = df["y"].value_counts().reindex(["no", "yes"]).rename("records").to_frame()
target_summary["percent"] = target_summary["records"].div(len(df)).mul(100)
target_summary.index.name = "y"
display(target_summary)
print(f"Observed subscription rate: {subscribed.mean():.2%}")
print("The outcome is imbalanced; accuracy alone would be an inadequate future evaluation metric.")

,records,percent
y,,
no,39922,88.30
yes,5289,11.70


Observed subscription rate: 11.70%
The outcome is imbalanced; accuracy alone would be an inadequate future evaluation metric.


## 4. Contact-history segments

`previous` counts contacts **before the current campaign**:

- `previous == 0`: no prior-campaign contact history.
- `previous > 0`: prior-campaign contact history.

These labels do not mean new versus existing bank customers. Segment assignment uses `previous`, not `poutcome`, because an unknown outcome is not evidence of no prior contact.

In [6]:
previous = df["previous"]
# Reject missing or invalid counts before the threshold could assign them to a history group.
assert previous.notna().all(), "Missing previous values need review."
assert previous.ge(0).all() and previous.mod(1).eq(0).all(), "Invalid previous counts need review."

segment_order = ["No prior-campaign contact", "Prior-campaign contact"]
# Prior-campaign contact defines these segments, not new versus existing bank customers.
history_segment = previous.gt(0).map({False: segment_order[0], True: segment_order[1]})
# assign returns a separate analysis frame, leaving the original loaded data unchanged.
analysis = df.assign(history_segment=history_segment, subscribed=subscribed)
segment_summary = analysis.groupby("history_segment", sort=False).agg(
    records=("y", "size"),
    subscriptions=("subscribed", "sum"),
    subscription_rate=("subscribed", "mean"),
).reindex(segment_order)
# Group share uses all records as its denominator; subscription rate uses only that group's records.
segment_summary["share_percent"] = segment_summary["records"].div(len(df)).mul(100)
segment_summary["subscription_percent"] = segment_summary.pop("subscription_rate").mul(100)
display(segment_summary)
print("Differences between groups are observational; they do not estimate the benefit of prior contact.")

,records,subscriptions,share_percent,subscription_percent
history_segment,,,,
No prior-campaign contact,36954,3384,81.74,9.16
Prior-campaign contact,8257,1905,18.26,23.07


Differences between groups are observational; they do not estimate the benefit of prior contact.


## 5. Prior outcome and history consistency

Review `poutcome` within the two history segments. In the prior-contact segment, `unknown` must remain distinct from `failure`; preserve `other` as its own recorded category too.

No-history rows can have `poutcome="unknown"` because no prior campaign outcome exists. That is structurally different from a prior-contact row whose result is unknown. The cross-tab is descriptive, not a rule for dropping or relabeling records.

In [7]:
# Keep 'unknown' and 'other' distinct from failure; previous=0 identifies structural no-history rows.
outcome_order = ["failure", "other", "success", "unknown"]
assert set(df["poutcome"].unique()) == set(outcome_order)
prior_counts = pd.crosstab(history_segment, df["poutcome"]).reindex(
    index=segment_order, columns=outcome_order, fill_value=0
)
prior_counts.index.name = "history_segment"
print("Recorded prior outcomes: counts")
display(prior_counts)
print("Recorded prior outcomes: percent within each segment")
# Divide by each row total (axis=0) to compare outcome composition within each history segment.
display(prior_counts.div(prior_counts.sum(axis=1), axis=0).mul(100))

# pdays=-1 is a source sentinel checked for consistency only; pdays stays excluded from primary models.
history_checks = pd.Series({
    "previous=0 but pdays is not -1": int((previous.eq(0) & df["pdays"].ne(-1)).sum()),
    "previous>0 but pdays is -1": int((previous.gt(0) & df["pdays"].eq(-1)).sum()),
    "previous=0 with a non-unknown poutcome": int((previous.eq(0) & df["poutcome"].ne("unknown")).sum()),
    "previous>0 with unknown poutcome": int((previous.gt(0) & df["poutcome"].eq("unknown")).sum()),
}, name="records").to_frame()
display(history_checks)
print("The five prior-contact rows with unknown outcomes stay distinct from recorded failures.")

Recorded prior outcomes: counts


poutcome,failure,other,success,unknown
history_segment,,,,
No prior-campaign contact,0,0,0,36954
Prior-campaign contact,4901,1840,1511,5


Recorded prior outcomes: percent within each segment


poutcome,failure,other,success,unknown
history_segment,,,,
No prior-campaign contact,0.00,0.00,0.00,100.00
Prior-campaign contact,59.36,22.28,18.30,0.06


,records
previous=0 but pdays is not -1,0
previous>0 but pdays is -1,0
previous=0 with a non-unknown poutcome,0
previous>0 with unknown poutcome,5


The five prior-contact rows with unknown outcomes stay distinct from recorded failures.


## 6. Numeric ranges for review

Inspect ranges before deciding on any cleaning. A large value is a review flag, not proof of an error. For example, `balance` can be negative; `previous` has a long upper tail. No clipping, deletion, or transformation is performed here.

The table includes excluded fields for data understanding only. Their appearance in a summary does not make them eligible predictors.

In [8]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()
# Tail percentiles flag values for review without clipping, dropping, or labeling them as errors.
display(df[numeric_columns].describe(percentiles=[0.5, 0.95, 0.99]).T)
range_flags = pd.Series({
    "negative balance": int(df["balance"].lt(0).sum()),
    "zero recorded duration": int(df["duration"].eq(0).sum()),
    "maximum previous contacts": int(df["previous"].max()),
    "maximum current-campaign contacts": int(df["campaign"].max()),
}, name="value").to_frame()
display(range_flags)

,count,mean,std,min,50%,95%,99%,max
age,"45,211.00",40.94,10.62,18.00,39.00,59.00,71.00,95.00
balance,"45,211.00","1,362.27","3,044.77","-8,019.00",448.00,"5,768.00","13,164.90","102,127.00"
day,"45,211.00",15.81,8.32,1.00,16.00,29.00,31.00,31.00
duration,"45,211.00",258.16,257.53,0.00,180.00,751.00,"1,269.00","4,918.00"
campaign,"45,211.00",2.76,3.10,1.00,2.00,8.00,16.00,63.00
pdays,"45,211.00",40.20,100.13,-1.00,-1.00,317.00,370.00,871.00
previous,"45,211.00",0.58,2.30,0.00,0.00,3.00,8.90,275.00


,value
negative balance,3766
zero recorded duration,3
maximum previous contacts,275
maximum current-campaign contacts,63


## 7. Exact duplicates and the limits of row-level checks

Count repeated full rows and repeated input profiles without changing the data. These checks answer whether the exported values repeat exactly; they cannot determine whether a person appears more than once with different values.

Even if both counts are zero, do not claim that there are 45,211 unique people or that a future split is customer-disjoint. The source provides no customer ID. The DataFrame index is just row position, not an identifier or a predictor.

In [9]:
# Ignore y in the profile check so identical inputs with different outcomes would also be detected.
input_columns = [column for column in df.columns if column != "y"]
# keep='first' counts extra copies; keep=False counts every member of a duplicate group.
# Neither check identifies repeated customers when their recorded values differ.
duplicate_summary = pd.Series({
    "rows": len(df),
    "extra exact full-row duplicates": int(df.duplicated(keep="first").sum()),
    "all rows in exact full-row duplicate groups": int(df.duplicated(keep=False).sum()),
    "extra identical input profiles, ignoring y": int(df.duplicated(subset=input_columns, keep="first").sum()),
}, name="records").to_frame()
display(duplicate_summary)
print("No records have been removed. Unique exported rows do not establish unique customers.")

,records
rows,45211
extra exact full-row duplicates,0
all rows in exact full-row duplicate groups,0
"extra identical input profiles, ignoring y",0


No records have been removed. Unique exported rows do not establish unique customers.


## 8. Prediction-time decisions

The intended decision point is **before the first call of the current campaign**. The table below documents candidate roles, not proof that the historical snapshots were captured at that time. Client attributes and prior-history fields still need timing review.

- Exclude `day`, `month`, `duration`, `campaign`, and `contact` from primary models because these describe the recorded current/last-contact process rather than a verified pre-first-call snapshot.
- Exclude **`pdays` from every primary model**. Its reference date is not established as the start of the current campaign. Any later use would require a separately justified sensitivity analysis.
- `previous` and `poutcome` are prior-campaign candidates, subject to timing verification. Unknown prior outcomes are not failures.
- `y` is the outcome, never an input feature. No association-driven feature selection is performed in this notebook.

In [10]:
# Roles follow the pre-first-call design, not correlations or performance measured against y.
common_candidates = ["age", "job", "marital", "education", "default", "balance", "housing", "loan"]
history_candidates = ["previous", "poutcome"]
excluded_reasons = {
    "contact": "Recorded current/last-contact channel; pre-first-call availability not established.",
    "day": "Day of the recorded last contact, not a known campaign-start snapshot.",
    "month": "Month of the recorded last contact, not a known campaign-start snapshot.",
    "duration": "Length of the recorded last call; unavailable before that call.",
    "campaign": "Current-campaign contact total including the last contact.",
    "pdays": "Recency reference date is not established as campaign start; excluded from primary models.",
}
# Require coverage of every input and prevent an excluded field from entering the candidate lists.
assert set(common_candidates + history_candidates + list(excluded_reasons)) == set(input_columns)
assert not set(common_candidates + history_candidates).intersection(excluded_reasons)

role_records = []
for feature in input_columns:
    if feature in common_candidates:
        role, reason = "Common candidate; verify timing", "Client attribute; confirm the recorded value was available before the first call."
    elif feature in history_candidates:
        role, reason = "History candidate; verify timing", "Prior-campaign information; verify timing and preserve unknown/no-history distinctions."
    else:
        role, reason = "Excluded from primary models", excluded_reasons[feature]
    role_records.append({"field": feature, "primary_role": role, "reason": reason})
display(pd.DataFrame(role_records).set_index("field"))
print("No feature matrix or model has been created.")

,primary_role,reason
field,,
age,Common candidate; verify timing,Client attribute; confirm the recorded value w...
job,Common candidate; verify timing,Client attribute; confirm the recorded value w...
marital,Common candidate; verify timing,Client attribute; confirm the recorded value w...
education,Common candidate; verify timing,Client attribute; confirm the recorded value w...
default,Common candidate; verify timing,Client attribute; confirm the recorded value w...
balance,Common candidate; verify timing,Client attribute; confirm the recorded value w...
housing,Common candidate; verify timing,Client attribute; confirm the recorded value w...
loan,Common candidate; verify timing,Client attribute; confirm the recorded value w...
contact,Excluded from primary models,Recorded current/last-contact channel; pre-fir...


No feature matrix or model has been created.


## 9. What this review does and does not establish

**The three future comparisons remain:** pooled common features, pooled features plus verified contact history, and separate history-segment models. Logistic Regression, Random Forest, and XGBoost are candidates, not fitted models.

Before modeling:

1. Resolve feature timing and define preprocessing using training data only.
2. Preserve the source's documented chronological order. Do not sort only by `month` and `day` to invent multi-year dates.
3. Fix one later chronological holdout for all three approaches, then partition each split by history segment. Review segment coverage before committing to a cutoff.
4. Use earlier data for tuning, calibration, and threshold selection; compare the same outreach budget overall and within segments.
5. Record that this initial descriptive review used the full source, including outcomes. It provides no independent model-validation result. Use genuinely unseen later data if an untouched external evaluation is required.

The dataset covers historically contacted clients. It cannot establish performance on all bank clients, the causal effect of contacting someone, or incremental conversion. Without verified feature timestamps, it may support only a leakage-aware retrospective comparison rather than a deployment-ready pre-call system.

## 10. Possible wrap-up: descriptive contact counts

A later, optional wrap-up could summarize observed subscription rates by current-campaign contact count (`campaign`), with group sizes and uncertainty. It is intentionally **not performed here** and is not part of the primary prediction feature set.

Contact counts reflect selection, repeat attempts, and stopping decisions. Such a descriptive association cannot show that one extra call caused a subscription, establish diminishing causal returns, or identify an optimal number of calls. `previous` and `campaign` also refer to different periods and must not be treated as interchangeable counts.

## 11. Takeaways and next decisions

- The expected original file loads with 45,211 records and 17 columns. Its observed subscription rate is about 11.70%.
- No parsed nulls are present, but literal `unknown` categories carry missing-information or no-history meaning.
- The contact-history groups contain 36,954 and 8,257 records. Their observed subscription rates differ; that does not show a causal benefit of previous contact.
- Five prior-contact records have an unknown prior outcome. Keep them distinct from failures and from no prior contact.
- No exact full-row duplicates or identical input profiles were found. Repeated people still cannot be ruled out without an identifier.
- Primary feature exclusions and unresolved timing are documented above. No rows have been dropped, no holdout has been selected, and no model has been trained.

The next review should agree on the feature dictionary, handling of unknown/no-history categories, and chronological evaluation plan before implementing preprocessing or models.

In [11]:
# Check in-memory values/order separately from on-disk bytes; both must remain unchanged.
pd.testing.assert_frame_equal(df, loaded_snapshot)
for relative_path, expected_hash in raw_hashes_before.items():
    assert sha256(PROJECT_ROOT / relative_path) == expected_hash, f"Raw file changed: {relative_path}"

print("PASS: raw files still match their initial SHA-256 hashes.")
print("PASS: the loaded DataFrame and source row order are unchanged.")
print("Completed data understanding only: no downloads, raw-file writes, cleaning exports, train/test split, or model fitting.")

PASS: raw files still match their initial SHA-256 hashes.
PASS: the loaded DataFrame and source row order are unchanged.
Completed data understanding only: no downloads, raw-file writes, cleaning exports, train/test split, or model fitting.
